# Merging Datasets and Unifying them into A Combined Dataset

This script merges the 2 open payments datasets together, merges the open payments with the DME Referring Provider Dataset, then it merges the LEIE list, then it merges the specialty information. 

This dataset feature engineers and pivots the aforementioned data to make sure they are all at the same grain: year and NPI. 

Careful consideration to joins and validate="one_to_one" on every merge was performed to ensure there were no cross products or unintentional duplicates.

Datasets required are:
- /dsa/groups/casestudycf25/team02/general_payments_products_clean.csv
- /dsa/groups/casestudycf25/team02/general_payments-sliced_clean.csv
- /dsa/groups/casestudycf25/team02/ownership_payment_clean.csv
- /dsa/groups/casestudycf25/team02/DMEPOS_Amount_Stats_labeled.csv
- /dsa/groups/casestudycf25/team02/one_specialty_per_npi.csv
- Note: Medicare Enrollment and Census data is not yet added. 


Feature Engineering and Cleaning
- Convert dataset cols to snake case
- Pivot General Payments on product_indicator to get counts per device type per record_id
- Map General Payments No/Yes columns to binary 0 and 1
- Create counts and sum columns and min/max date cols for General Payments to get a dataset at the ('covered_recipient_npi', 'program_year', 'applicable_manufacturer_or_applicable_gpo_making_payment_id') level.
- Group the Ownership data by ('covered_recipient_npi','program_year','applicable_manufacturer_or_applicable_gpo_making_payment_id') and aggregate the 
        total_invested_usd=('total_amount_invested_us_dollars', 'sum'),
        total_value_of_interest=('value_of_interest', 'sum'),
        n_owner_records=('record_id', 'nunique').
- Pivot the Ownership Data by the interest_held_by_physician_or_an_immediate_family_member to create two new cols
- Left Join the General Payments and the Ownership data (keeping all General Payments) together by ('covered_recipient_npi','program_year','applicable_manufacturer_or_applicable_gpo_making_payment_id') 
- Group Open Payments to yield a dataset at NPI + program year. Summing, counting manufacturing ID. And then min-maxing the payment dates.
- Left Join the DMEPOS Referring Provider Dataset and the All Payments Dataset, keeping all rows from DMEPOS. Keep at the NPI and Year level.
- Left Join the Unified Dataset with the LEIE based on the condition dme_year <= leie_year
- Inner Join the Unified Dataset with the one_specialty_per_npi.csv dataset to get specialty hierarchy info.

This script outputs interim CSVs as the merges and transformations take place.

It returns one dataset called unified_dataset.csv to the /dsa/groups/casestudycf25/team02/ folder.

In [2]:
# function to convert all datasets to snake_case
import re

def to_snake_case(name: str) -> str:
    # Add underscore between lower-to-upper transitions
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)

    # Replace non-alphanumeric with underscores
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)

    # Remove leading/trailing underscores and lowercase
    return name.strip("_").lower()

# Loading Data

In [140]:
import pandas as pd
import numpy as np

### General Payments

In [3]:
products = pd.read_csv("/dsa/groups/casestudycf25/team02/general_payments_products_clean.csv")

In [4]:
products.shape

(7982139, 8)

In [5]:
products["product_category"].unique()

array(['BONE SUBSTITUTES', 'TRAUMA & EXTREMITIES', 'SPINE', ...,
       'IMRPESSION MATERIAL',
       'KNEE & HIP - IMPLANTS - KNEE ANCHORS - KNEE FIBERTAK',
       'BIOLOGICS - CONSUMABLES - AUTOLOGOUS CELLULAR PRODUCTS - ACP PRP'],
      dtype=object)

In [6]:
products.head()

,Unnamed: 0,record_id,pdi,product_name,covered_indicator,product_category,drug_or_device_indicator,slot
0,0,1006722961,10859477002140,AUGMENT INJECTABLE,Covered,BONE SUBSTITUTES,Device,1
1,1,1006722977,00889797103404,EASYFUSE,Covered,TRAUMA & EXTREMITIES,Device,1
2,2,1006723021,00840420121912,INFINITY,Covered,TRAUMA & EXTREMITIES,Device,1
3,3,1006723073,00840420123022,PRO-TOE,Covered,TRAUMA & EXTREMITIES,Device,1
4,4,1006530267,10888857031555,CAPRI CORPECTOMY CAGE SYSTEM,Covered,SPINE,Device,1


In [7]:
products["product_indicator"] = products["covered_indicator"].str.lower() + "_"+ products["drug_or_device_indicator"].str.lower()

In [8]:
# Pivot the products dataframe to join to the sliced one
pivot = (
    pd.crosstab(products['record_id'], products['product_indicator'])
    .reset_index()
)

print(pivot.head())

product_indicator  record_id  covered_biological  covered_device  \
0                  756635078                   0               0   
1                  756703674                   0               0   
2                  756759262                   0               0   
3                  756772330                   0               0   
4                  756861474                   0               0   

product_indicator  covered_drug  covered_medical supply  non-covered_device  \
0                             0                       0                   1   
1                             0                       0                   1   
2                             0                       0                   1   
3                             0                       0                   1   
4                             0                       0                   1   

product_indicator  non-covered_drug  non-covered_medical supply  
0                                 0               

In [9]:
# Get sums for each product_indicator column
col_sums = pivot.drop(columns=['record_id']).sum(axis=0)

print(col_sums)

product_indicator
covered_biological               5501
covered_device                7505817
covered_drug                   107754
covered_medical supply         226364
non-covered_device             128992
non-covered_drug                 7376
non-covered_medical supply        322
dtype: int64


In [10]:
gp = pd.read_csv("/dsa/groups/casestudycf25/team02/general_payments-sliced_clean.csv")

In [11]:
gp.columns

Index(['Unnamed: 0', 'record_id', 'change_type', 'payment_publication_date',
       'date_of_payment', 'program_year', 'covered_recipient_npi',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'total_amount_of_payment_us_dollars',
       'number_of_payments_included_in_total_amount',
       'form_of_payment_or_transfer_of_value',
       'nature_of_payment_or_transfer_of_value', 'nature_short_descr',
       'city_of_travel', 'state_of_travel', 'country_of_travel',
       'physician_ownership_indicator',
       'third_party_payment_recipient_indicator',
       'name_of_third_party_entity_receiving_payment_or_transfer_of_value',
       'charity_indicator', 'third_party_equals_covered_recipient_indicator',
       'dispute_status_for_publication'],
      dtype='object')

In [12]:
gp.shape

(6562870, 22)

In [13]:
###################################
# INNER JOIN General Payments Transactions to Service Pivot
###################################
merged_gp = pd.merge(gp, pivot, right_on = "record_id", left_on = "record_id", how = "inner", validate="one_to_one")

merged_gp.shape

(6467206, 29)

In [14]:
merged_gp.head(2)

,Unnamed: 0,record_id,change_type,payment_publication_date,date_of_payment,program_year,covered_recipient_npi,applicable_manufacturer_or_applicable_gpo_making_payment_id,total_amount_of_payment_us_dollars,number_of_payments_included_in_total_amount,...,charity_indicator,third_party_equals_covered_recipient_indicator,dispute_status_for_publication,covered_biological,covered_device,covered_drug,covered_medical supply,non-covered_device,non-covered_drug,non-covered_medical supply
0,0,1006722961,UNCHANGED,2025-06-30,2023-05-23,2023,1497734156,100000010503,20.97,1,...,NaN,NaN,No,0,1,0,0,0,0,0
1,1,1006722977,UNCHANGED,2025-06-30,2023-07-06,2023,1497734156,100000010503,10.89,1,...,NaN,NaN,No,0,1,0,0,0,0,0


In [15]:
########################
# Mapping Text columns to numbers
#############################
mapping = {'No': 0, 'Yes': 1}

In [16]:
merged_gp["charity_indicator"] = (
      merged_gp["charity_indicator"]
      .map(mapping)     # map Yes/No
      .fillna(0)       # turn NaN into 0
      .astype('int8')
)

In [ ]:
# merged_gp["physician_ownership_indicator"] = (
#       merged_gp["physician_ownership_indicator"]
#       .map(mapping)     # map Yes/No
#       .fillna(0)       # turn NaN into 0
#       .astype('int8')
# )

In [17]:
merged_gp["third_party_equals_covered_recipient_indicator"]= (
      merged_gp["third_party_equals_covered_recipient_indicator"]
      .map(mapping)     # map Yes/No
      .fillna(0)       # turn NaN into -1
      .astype('int8')
)

In [18]:
merged_gp["dispute_status_for_publication"]= (
      merged_gp["dispute_status_for_publication"]
      .map(mapping)     # map Yes/No
      .fillna(0)       # turn NaN into 0
      .astype('int8')
)

In [19]:
merged_gp.head(1)

,Unnamed: 0,record_id,change_type,payment_publication_date,date_of_payment,program_year,covered_recipient_npi,applicable_manufacturer_or_applicable_gpo_making_payment_id,total_amount_of_payment_us_dollars,number_of_payments_included_in_total_amount,...,charity_indicator,third_party_equals_covered_recipient_indicator,dispute_status_for_publication,covered_biological,covered_device,covered_drug,covered_medical supply,non-covered_device,non-covered_drug,non-covered_medical supply
0,0,1006722961,UNCHANGED,2025-06-30,2023-05-23,2023,1497734156,100000010503,20.97,1,...,0,0,0,0,1,0,0,0,0,0


In [20]:
# Output general_payments_record_id_level_clean

merged_gp.to_csv("general_payments_record_id_level_clean.csv")

In [21]:
merged_gp.columns

Index(['Unnamed: 0', 'record_id', 'change_type', 'payment_publication_date',
       'date_of_payment', 'program_year', 'covered_recipient_npi',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'total_amount_of_payment_us_dollars',
       'number_of_payments_included_in_total_amount',
       'form_of_payment_or_transfer_of_value',
       'nature_of_payment_or_transfer_of_value', 'nature_short_descr',
       'city_of_travel', 'state_of_travel', 'country_of_travel',
       'physician_ownership_indicator',
       'third_party_payment_recipient_indicator',
       'name_of_third_party_entity_receiving_payment_or_transfer_of_value',
       'charity_indicator', 'third_party_equals_covered_recipient_indicator',
       'dispute_status_for_publication', 'covered_biological',
       'covered_device', 'covered_drug', 'covered_medical supply',
       'non-covered_device', 'non-covered_drug', 'non-covered_medical supply'],
      dtype='object')

In [22]:
###############
# Group the General Payments to Program Year, NPI, and Manufacturing ID

# merged_gp['form_of_payment_or_transfer_of_value'].unique() #categorical
# merged_gp['third_party_payment_recipient_indicator'].unique() #categorical
# merged_gp['nature_short_descr']  #categorical
# merged_gp['name_of_third_party_entity_receiving_payment_or_transfer_of_value'].nunique() # unique names
# applicable_manufacturer_or_applicable_gpo_making_payment_id #unique ids
# merged_gp["charity_indicator"] #binary
# merged_gp["third_party_equals_covered_recipient_indicator"] #binary
# merged_gp["dispute_status_for_publication"] #binary

# #counts
# ['covered_biological','covered_device', 'covered_drug', 'covered_medical supply',
#  'non-covered_device', 'non-covered_drug', 'non-covered_medical supply'] 

In [23]:
merge_keys = [
    'covered_recipient_npi',
    'program_year',
    'applicable_manufacturer_or_applicable_gpo_making_payment_id'
]
value_col = 'total_amount_of_payment_us_dollars'

In [24]:
def make_count_and_sum(
    df: pd.DataFrame,
    group_cols,
    cat_col: str,
    value_col: str,
    count_prefix: str,
    sum_prefix: str,
):
    """
    Returns:
      counts_df: one column per category with counts
      sums_df:   one column per category with dollar sums
    """
    tmp = (
        df
        .groupby(group_cols + [cat_col])
        .agg(
            count=('record_id', 'size'),
            total_sum=(value_col, 'sum'),
        )
        .reset_index()
    )

    counts = tmp.pivot_table(
        index=group_cols,
        columns=cat_col,
        values='count',
        fill_value=0
    )

    sums = tmp.pivot_table(
        index=group_cols,
        columns=cat_col,
        values='total_sum',
        fill_value=0.0
    )

    counts = counts.rename(
        columns=lambda c: f"{count_prefix}_{to_snake_case(str(c))}"
    )
    sums = sums.rename(
        columns=lambda c: f"{sum_prefix}_{to_snake_case(str(c))}"
    )

    counts = counts.reset_index()
    sums = sums.reset_index()

    return counts, sums

In [25]:
####################
# Create Count and Sum columns for each of the categorical cols
######################

# Form of payment
form_counts, form_sums = make_count_and_sum(
    merged_gp,
    group_cols=merge_keys,
    cat_col='form_of_payment_or_transfer_of_value',
    value_col=value_col,
    count_prefix='form_count',
    sum_prefix='form_sum'
)

# Third-party payment recipient indicator
tpp_counts, tpp_sums = make_count_and_sum(
    merged_gp,
    group_cols=merge_keys,
    cat_col='third_party_payment_recipient_indicator',
    value_col=value_col,
    count_prefix='third_party_recipient_count',
    sum_prefix='third_party_recipient_sum'
)

# Nature short description
nature_counts, nature_sums = make_count_and_sum(
    merged_gp,
    group_cols=merge_keys,
    cat_col='nature_short_descr',
    value_col=value_col,
    count_prefix='nature_count',
    sum_prefix='nature_sum'
)

In [26]:
third_party_summary = (
    merged_gp
    .groupby(merge_keys)
    .agg(
        n_third_party_entities=(
            'name_of_third_party_entity_receiving_payment_or_transfer_of_value',
            'nunique'
        )
    )
    .reset_index()
)

In [27]:
numeric_gp_cols = [
    
    # binary cols
    'physician_ownership_indicator',
    'charity_indicator',
    'third_party_equals_covered_recipient_indicator',
    'dispute_status_for_publication',
    
    
    # count and monetary cols
    'total_amount_of_payment_us_dollars',
    'number_of_payments_included_in_total_amount',
    'covered_biological',
    'covered_device',
    'covered_drug',
    'covered_medical supply',
    'non-covered_device',
    'non-covered_drug',
    'non-covered_medical supply'
]

numeric_gp_agg = (
    merged_gp
    .groupby(merge_keys)[numeric_gp_cols]
    .sum()
    .reset_index()
)

In [28]:
record_counts = (
    merged_gp
    .groupby(merge_keys)
    .size()
    .reset_index(name='n_records')
)

In [29]:
date_agg = (
    merged_gp
    .groupby(merge_keys)
    .agg(
        first_payment_date=('date_of_payment', 'min'),
        last_payment_date=('date_of_payment', 'max')
    )
    .reset_index()
)

geo_agg = (
    merged_gp
    .groupby(merge_keys)
    .agg(
        n_cities=('city_of_travel', 'nunique'),
        n_states=('state_of_travel', 'nunique'),
        n_countries=('country_of_travel', 'nunique'),
    )
    .reset_index()
)

In [30]:
###########
# Drop these columns, unnecessary for general payments agg df
###########
cols_to_drop = [
    'Unnamed: 0',
    'change_type',
    'payment_publication_date', 
    'nature_of_payment_or_transfer_of_value'
]

merged_gp = merged_gp.drop(columns=cols_to_drop)

In [31]:
dfs_to_merge = [
    record_counts,
    numeric_gp_agg,
    date_agg,
    geo_agg,
    form_counts, form_sums,
    tpp_counts, tpp_sums,
    nature_counts, nature_sums,
    third_party_summary
]

In [32]:
# The aggregated dataframes of General Payments from all the pivoting and grouping must be merged! The list is below.
for df in dfs_to_merge:
    print(df.shape)

(2256621, 4)
(2256621, 15)
(2256621, 5)
(2256621, 6)
(2256621, 9)
(2256621, 9)
(2256621, 6)
(2256621, 6)
(2209529, 13)
(2209529, 13)
(2256621, 4)


In [33]:
from functools import reduce

# Merge aggregated dataframes of General Payments. 
# Based on 'covered_recipient_npi', 'program_year', 'applicable_manufacturer_or_applicable_gpo_making_payment_id'
gp_summary = reduce(
    lambda left, right: left.merge(right, on=merge_keys, how='left'),
    dfs_to_merge
)

In [34]:
gp_summary.shape

(2256621, 60)

In [35]:
# Delete the temp agg tables
del (numeric_gp_agg, 
     date_agg, geo_agg,
    form_counts, form_sums,
    tpp_counts, tpp_sums,
    nature_counts, nature_sums,
    third_party_summary,
    dfs_to_merge, gp, merged_gp, record_counts)

In [36]:
gp_summary.columns

Index(['covered_recipient_npi', 'program_year',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'n_records', 'charity_indicator',
       'third_party_equals_covered_recipient_indicator',
       'dispute_status_for_publication', 'total_amount_of_payment_us_dollars',
       'number_of_payments_included_in_total_amount', 'covered_biological',
       'covered_device', 'covered_drug', 'covered_medical supply',
       'non-covered_device', 'non-covered_drug', 'non-covered_medical supply',
       'first_payment_date', 'last_payment_date', 'n_cities', 'n_states',
       'n_countries', 'form_count_cash_or_cash_equivalent',
       'form_count_dividend_profit_or_other_return_on_investment',
       'form_count_in_kind_items_and_services', 'form_count_stock',
       'form_count_stock_option',
       'form_count_stock_stock_option_or_any_other_ownership_interest',
       'form_sum_cash_or_cash_equivalent',
       'form_sum_dividend_profit_or_other_return_on_investment',


In [37]:
gp_summary.head()

,covered_recipient_npi,program_year,applicable_manufacturer_or_applicable_gpo_making_payment_id,n_records,charity_indicator,third_party_equals_covered_recipient_indicator,dispute_status_for_publication,total_amount_of_payment_us_dollars,number_of_payments_included_in_total_amount,covered_biological,...,nature_sum_charity,nature_sum_consulting,nature_sum_debt_forgiveness,nature_sum_education,nature_sum_entertainment,nature_sum_faculty_or_speaker,nature_sum_food_and_beverage,nature_sum_other_services,nature_sum_ownership_or_investment,n_third_party_entities
0,1003000142,2021,100000000106,1,0,0,0,168.00,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,168.00,0.0,0.0,0
1,1003000142,2021,100000005674,10,0,0,0,273.20,10,0,...,0.0,0.0,0.0,0.0,0.0,0.0,273.20,0.0,0.0,0
2,1003000142,2021,100000010680,7,0,0,0,226.09,7,0,...,0.0,0.0,0.0,0.0,0.0,0.0,226.09,0.0,0.0,0
3,1003000142,2021,100000016247,4,0,0,0,232.31,4,0,...,0.0,0.0,0.0,0.0,0.0,0.0,232.31,0.0,0.0,0
4,1003000142,2021,100000136428,3,0,0,0,46.59,3,0,...,0.0,0.0,0.0,0.0,0.0,0.0,46.59,0.0,0.0,0


In [38]:
gp_summary.describe()

,covered_recipient_npi,program_year,applicable_manufacturer_or_applicable_gpo_making_payment_id,n_records,charity_indicator,third_party_equals_covered_recipient_indicator,dispute_status_for_publication,total_amount_of_payment_us_dollars,number_of_payments_included_in_total_amount,covered_biological,...,nature_sum_charity,nature_sum_consulting,nature_sum_debt_forgiveness,nature_sum_education,nature_sum_entertainment,nature_sum_faculty_or_speaker,nature_sum_food_and_beverage,nature_sum_other_services,nature_sum_ownership_or_investment,n_third_party_entities
count,2.256621e+06,2.256621e+06,2.256621e+06,2.256621e+06,2.256621e+06,2.256621e+06,2.256621e+06,2.256621e+06,2.256621e+06,2.256621e+06,...,2.209529e+06,2.209529e+06,2.209529e+06,2.209529e+06,2.209529e+06,2.209529e+06,2.209529e+06,2.209529e+06,2.209529e+06,2.256621e+06
mean,1.499525e+09,2.022109e+03,1.000001e+11,2.865880e+00,5.893768e-05,7.361006e-03,1.050243e-04,1.361695e+03,2.909080e+00,2.437716e-03,...,8.699421e-02,2.443893e+02,2.466097e+00,5.545265e+01,1.880839e-01,2.102959e+01,1.111093e+02,9.648752e+01,3.559089e+00,1.047008e-02
std,2.880940e+08,8.099053e-01,2.871996e+05,6.676825e+00,1.953306e-02,2.374097e-01,1.685355e-02,6.292405e+04,6.937217e+00,7.928563e-02,...,5.352825e+01,5.061752e+03,3.367136e+02,1.435829e+03,1.146492e+01,2.296917e+03,3.104033e+02,2.905560e+03,1.233766e+03,1.288268e-01
min,1.003000e+09,2.021000e+03,1.000000e+11,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.100000e-01,1.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.245725e+09,2.021000e+03,1.000000e+11,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.117000e+01,1.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.972000e+01,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.497976e+09,2.022000e+03,1.000000e+11,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.986000e+01,1.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.297000e+01,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.750320e+09,2.023000e+03,1.000000e+11,2.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.426900e+02,2.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.250000e+02,0.000000e+00,0.000000e+00,0.000000e+00
max,1.993000e+09,2.023000e+03,1.000015e+11,7.910000e+02,2.300000e+01,9.000000e+01,1.300000e+01,3.328394e+07,1.308000e+03,4.200000e+01,...,5.341000e+04,2.391608e+06,2.427825e+05,2.834098e+05,5.742650e+03,2.520955e+06,8.870275e+04,1.967159e+06,1.000000e+06,2.700000e+01


In [39]:
# Output general_payments_record_id_level_clean

gp_summary.to_csv("general_payments_npi_year_manufacturing_clean.csv")

### Ownership Payments

In [40]:
ownership = pd.read_csv("/dsa/groups/casestudycf25/team02/ownership_payment_clean.csv")

In [41]:
ownership.shape

(12710, 15)

In [42]:
ownership.head()

,Change_Type,Physician_Profile_ID,Physician_NPI,Record_ID,Program_Year,Total_Amount_Invested_USDollars,Value_of_Interest,Terms_of_Interest,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name,Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_State,Dispute_Status_for_Publication,Interest_Held_by_Physician_or_an_Immediate_Family_Member,Payment_Publication_Date,FULL_NAME
0,UNCHANGED,361157,1.962476e+09,754966674,2021,0.0,296.69,EQUITY DISTRIBUTION PAYMENT,100000011188,"MOBIUS THERAPEUTICS, LLC",MO,No,Physician Covered Recipient,6/30/2025,MARLENE MOSTER
1,UNCHANGED,173155,1.821079e+09,754966682,2021,0.0,593.37,EQUITY DISTRIBUTION PAYMENT,100000011188,"MOBIUS THERAPEUTICS, LLC",MO,No,Physician Covered Recipient,6/30/2025,ROBERT RITCH
2,NEW,84927,1.881709e+09,755268702,2021,0.0,31926.00,PRIVATE EQUITY,100000806858,"MANUAL SURGICAL SCIENCES, LLC",UT,No,Physician Covered Recipient,6/30/2025,ANDRELUIZ DAVILA
3,NEW,119199,1.639176e+09,755300384,2021,0.0,43161.00,PRIVATE EQUITY,100000806858,"MANUAL SURGICAL SCIENCES, LLC",UT,No,Physician Covered Recipient,6/30/2025,SHEPHAL DOSHI
4,NEW,233626,1.144259e+09,755306684,2021,0.0,87984.00,PRIVATE EQUITY,100000806858,"MANUAL SURGICAL SCIENCES, LLC",UT,No,Physician Covered Recipient,6/30/2025,SRINIVAS DUKKIPATI


In [43]:
ownership = ownership.rename(columns={col: to_snake_case(col) for col in ownership.columns})

In [44]:
ownership["interest_held_by_physician_or_an_immediate_family_member"].unique()

array(['Physician Covered Recipient', 'Immediate family member'],
      dtype=object)

In [46]:
ownership["covered_recipient_npi"] = ownership["physician_npi"]

In [47]:
summary = (
    ownership.groupby(['covered_recipient_npi',
                       'program_year',
                       'applicable_manufacturer_or_applicable_gpo_making_payment_id']).agg(
        total_invested_usd=('total_amount_invested_us_dollars', 'sum'),
        total_value_of_interest=('value_of_interest', 'sum'),
        n_owner_records=('record_id', 'nunique')
    )
).reset_index() 

summary.head()

,covered_recipient_npi,program_year,applicable_manufacturer_or_applicable_gpo_making_payment_id,total_invested_usd,total_value_of_interest,n_owner_records
0,1.003001e+09,2021,100000005385,0.0,38419.76,1
1,1.003001e+09,2022,100000005385,0.0,38419.76,1
2,1.003001e+09,2023,100000005385,0.0,38419.76,1
3,1.003013e+09,2021,100000966845,75000.0,169597.00,1
4,1.003013e+09,2022,100000966845,0.0,169590.00,1


In [48]:
summary.columns

Index(['covered_recipient_npi', 'program_year',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'total_invested_usd', 'total_value_of_interest', 'n_owner_records'],
      dtype='object')

In [49]:
# Pivot interest_held into one column per interest type (counts)
interest_pivot = (
    pd.crosstab(
        [ownership['covered_recipient_npi'], ownership['program_year'], ownership['applicable_manufacturer_or_applicable_gpo_making_payment_id']],
        ownership['interest_held_by_physician_or_an_immediate_family_member']
    )
    .reset_index()
)


interest_table = interest_pivot.groupby(['covered_recipient_npi', 'program_year',
                                         'applicable_manufacturer_or_applicable_gpo_making_payment_id'], 
                                        as_index=False)[['Immediate family member','Physician Covered Recipient']].sum()

interest_table = interest_table.rename(columns={col: to_snake_case(col) for col in interest_table.columns})

interest_table

interest_held_by_physician_or_an_immediate_family_member,covered_recipient_npi,program_year,applicable_manufacturer_or_applicable_gpo_making_payment_id,immediate_family_member,physician_covered_recipient
0,1.003001e+09,2021,100000005385,0,1
1,1.003001e+09,2022,100000005385,0,1
2,1.003001e+09,2023,100000005385,0,1
3,1.003013e+09,2021,100000966845,1,0
4,1.003013e+09,2022,100000966845,1,0
...,...,...,...,...,...
11934,1.992981e+09,2022,100000826852,0,1
11935,1.992981e+09,2023,100000826852,0,1
11936,1.992993e+09,2021,100000151617,0,1
11937,1.992993e+09,2022,100000151617,0,1


In [62]:
# Merge ownership interest pivot with ownership payments summary
ownership_final = summary.merge(
    interest_table,
    on=['covered_recipient_npi', 'program_year', 'applicable_manufacturer_or_applicable_gpo_making_payment_id'],
    how="inner",          
    validate="one_to_one" 
)
ownership_final.head()

NameError: name 'summary' is not defined

In [51]:
del ownership, interest_pivot, interest_table, summary

### Merging the Payments Tables Together

In [52]:
merge_keys = [
    'covered_recipient_npi',
    'program_year',
    'applicable_manufacturer_or_applicable_gpo_making_payment_id'
]

# In summary
dupes_gp_summary = gp_summary[gp_summary.duplicated(subset=merge_keys, keep=False)] \
    .sort_values(merge_keys)
print("Duplicates in summary:")
print(dupes_gp_summary.head(20))

# In interest_table
dupes_ownership = ownership_final[ownership_final.duplicated(subset=merge_keys, keep=False)] \
    .sort_values(merge_keys)
print("Duplicates in interest_table:")
print(dupes_ownership.head(20))


Duplicates in summary:
Empty DataFrame
Columns: [covered_recipient_npi, program_year, applicable_manufacturer_or_applicable_gpo_making_payment_id, n_records, charity_indicator, third_party_equals_covered_recipient_indicator, dispute_status_for_publication, total_amount_of_payment_us_dollars, number_of_payments_included_in_total_amount, covered_biological, covered_device, covered_drug, covered_medical supply, non-covered_device, non-covered_drug, non-covered_medical supply, first_payment_date, last_payment_date, n_cities, n_states, n_countries, form_count_cash_or_cash_equivalent, form_count_dividend_profit_or_other_return_on_investment, form_count_in_kind_items_and_services, form_count_stock, form_count_stock_option, form_count_stock_stock_option_or_any_other_ownership_interest, form_sum_cash_or_cash_equivalent, form_sum_dividend_profit_or_other_return_on_investment, form_sum_in_kind_items_and_services, form_sum_stock, form_sum_stock_option, form_sum_stock_stock_option_or_any_other_owne

In [63]:
# composite key columns
keys = ["covered_recipient_npi", "applicable_manufacturer_or_applicable_gpo_making_payment_id", "program_year"]

# Merging the General Payments final table with the ownership final table
all_payments = gp_summary.merge(
    ownership_final,
    on=keys,
    how="left",         
    validate="one_to_one"
)

In [64]:
all_payments.shape

(2256621, 65)

In [65]:
all_payments.to_csv("all_payments_npi_year_manufacturing_clean.csv")

In [66]:
all_payments.columns

Index(['covered_recipient_npi', 'program_year',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'n_records', 'charity_indicator',
       'third_party_equals_covered_recipient_indicator',
       'dispute_status_for_publication', 'total_amount_of_payment_us_dollars',
       'number_of_payments_included_in_total_amount', 'covered_biological',
       'covered_device', 'covered_drug', 'covered_medical supply',
       'non-covered_device', 'non-covered_drug', 'non-covered_medical supply',
       'first_payment_date', 'last_payment_date', 'n_cities', 'n_states',
       'n_countries', 'form_count_cash_or_cash_equivalent',
       'form_count_dividend_profit_or_other_return_on_investment',
       'form_count_in_kind_items_and_services', 'form_count_stock',
       'form_count_stock_option',
       'form_count_stock_stock_option_or_any_other_ownership_interest',
       'form_sum_cash_or_cash_equivalent',
       'form_sum_dividend_profit_or_other_return_on_investment',


### Group all Open Payments to NPI + program year

In [67]:
################
# - manufacturing counts
# - sum / mean payment
# - sums for all other numeric columns
################

group_cols = ["covered_recipient_npi", "program_year"]

# All numeric columns
numeric_cols = all_payments.select_dtypes(include="number").columns.tolist()

# Columns we do NOT want to auto-sum
exclude_from_auto_sum = set(
    group_cols
    + [
        "applicable_manufacturer_or_applicable_gpo_making_payment_id",  # get unique counts later
        "total_amount_of_payment_us_dollars",                           # handled separately (sum/mean)
    ]
)

numeric_sum_cols = [c for c in numeric_cols if c not in exclude_from_auto_sum]

# Build dict of sum aggs for the remaining numeric columns
sum_aggs = {c: (c, "sum") for c in numeric_sum_cols}

all_payments_npi_year_final = (
    all_payments
    .groupby(group_cols)
    .agg(
        # manufacturing-related
        n_manufacturing_entities=(
            "applicable_manufacturer_or_applicable_gpo_making_payment_id",
            "nunique"
        ),

        # dates
        first_payment_date=("first_payment_date", "min"),
        last_payment_date=("last_payment_date", "max"),

        # payment stats
        total_payment=("total_amount_of_payment_us_dollars", "sum"),
        mean_payment_manufacturers =("total_amount_of_payment_us_dollars", "mean"),

        # all other numeric sums
        **sum_aggs,
    )
    .reset_index()
)


In [68]:
all_payments_npi_year_final.to_csv("all_payments_npi_year_final_clean.csv")

In [69]:
all_payments_npi_year_final.shape

(1069530, 66)

In [71]:
del gp_summary, ownership_final

### DMEPOS

In [114]:
dme_year_npi = pd.read_csv('/dsa/groups/casestudycf25/team02/DMEPOS_Amount_Stats_labeled.csv',dtype={'Rfrg_Prvdr_State_FIPS':str,'Rfrg_Prvdr_Zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str


In [115]:
#######################
# SNAKE CASE DME 
dme_year_npi = dme_year_npi.rename(columns={col: to_snake_case(col) for col in dme_year_npi.columns})

In [116]:
dme_year_npi.shape

(879531, 94)

In [117]:
dme_year_npi.head()

,npi,year,target,avg_suplr_sbmtd_chrg_mean,avg_suplr_mdcr_alowd_amt_mean,avg_suplr_mdcr_pymt_amt_mean,avg_suplr_mdcr_stdzd_amt_mean,avg_suplr_sbmtd_chrg_sum,avg_suplr_mdcr_alowd_amt_sum,avg_suplr_mdcr_pymt_amt_sum,...,bene_cc_ph_diabetes_v2_pct,bene_cc_ph_hf_non_ihd_v2_pct,bene_cc_ph_hyperlipidemia_v2_pct,bene_cc_ph_hypertension_v2_pct,bene_cc_ph_ischemic_heart_v2_pct,bene_cc_ph_osteoporosis_v2_pct,bene_cc_ph_parkinson_v2_pct,bene_cc_ph_arthritis_v2_pct,bene_cc_ph_stroke_tia_v2_pct,bene_avg_risk_scre
0,1003000126,2021,0,129.776563,41.940392,31.813932,34.260562,519.106250,167.761567,127.255730,...,0.0,0.0,0.000000,1.000000,0.000000,0.0,0.0,0.0,0.0,1.942167
1,1003000126,2022,0,209.273634,54.750832,41.764741,52.602407,418.547268,109.501664,83.529481,...,0.0,0.0,0.833333,0.944444,0.777778,0.0,0.0,0.0,0.0,2.987489
2,1003000126,2023,0,152.346111,48.740210,32.699442,42.116461,457.038333,146.220631,98.098326,...,0.0,0.0,0.000000,1.000000,0.000000,0.0,0.0,0.0,0.0,3.500804
3,1003000480,2021,0,272.003846,80.513846,64.407692,84.701538,272.003846,80.513846,64.407692,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,2.278200
4,1003000522,2021,0,133.101129,22.805968,18.243952,19.471129,266.202258,45.611935,36.487903,...,0.0,0.0,1.000000,0.846154,0.000000,0.0,0.0,0.0,0.0,1.872308


In [118]:
# In summary
dupes_dme_year_npi = dme_year_npi[dme_year_npi.duplicated(subset=['npi', 'year'], keep=False)] \
    .sort_values(['npi', 'year'])
print("Duplicates in dme_year_npi:")
print(dupes_dme_year_npi.head(20))

# In payments table
dupes_all_payments = all_payments_npi_year_final[all_payments_npi_year_final.duplicated(subset= ['covered_recipient_npi','program_year']
                                                                                        , keep=False)] \
    .sort_values(['covered_recipient_npi','program_year'])
print("Duplicates in all_payments_npi_year_final:")
print(dupes_all_payments.head(20))

Duplicates in dme_year_npi:
Empty DataFrame
Columns: [npi, year, target, avg_suplr_sbmtd_chrg_mean, avg_suplr_mdcr_alowd_amt_mean, avg_suplr_mdcr_pymt_amt_mean, avg_suplr_mdcr_stdzd_amt_mean, avg_suplr_sbmtd_chrg_sum, avg_suplr_mdcr_alowd_amt_sum, avg_suplr_mdcr_pymt_amt_sum, avg_suplr_mdcr_stdzd_amt_sum, avg_suplr_sbmtd_chrg_median, avg_suplr_mdcr_alowd_amt_median, avg_suplr_mdcr_pymt_amt_median, avg_suplr_mdcr_stdzd_amt_median, avg_suplr_sbmtd_chrg_std, avg_suplr_mdcr_alowd_amt_std, avg_suplr_mdcr_pymt_amt_std, avg_suplr_mdcr_stdzd_amt_std, avg_suplr_sbmtd_chrg_min, avg_suplr_mdcr_alowd_amt_min, avg_suplr_mdcr_pymt_amt_min, avg_suplr_mdcr_stdzd_amt_min, avg_suplr_sbmtd_chrg_max, avg_suplr_mdcr_alowd_amt_max, avg_suplr_mdcr_pymt_amt_max, avg_suplr_mdcr_stdzd_amt_max, tot_suplr_nonrntl_hcpcs_cds, tot_suplr_rentl_hcpcs_cds, tot_suplrs_mean, tot_suplr_benes_mean, tot_suplr_clms_mean, tot_suplr_srvcs_mean, tot_suplrs_sum, tot_suplr_benes_sum, tot_suplr_clms_sum, tot_suplr_srvcs_sum, tot_s

In [119]:
# Merging the DME year and npi to the payments table
dme_payments_merge = dme_year_npi.merge(
    all_payments_npi_year_final,
    left_on =  ['npi', 'year'],
    right_on = ['covered_recipient_npi','program_year'],
    how="left",         
    validate="one_to_one"
)

In [120]:
dme_payments_merge.shape

(879531, 160)

In [121]:
# Number of NaNs in the column 'covered_recipient_npi'
n_nans_covered_recipient_npi = dme_payments_merge['covered_recipient_npi'].isna().sum()
print("NaNs in covered_recipient_npi:", n_nans_covered_recipient_npi)
nan_percent = dme_payments_merge['covered_recipient_npi'].isna().mean() * 100
print(f"Percentage of NaNs in covered_recipient_npi: {nan_percent:.2f}%")

NaNs in covered_recipient_npi: 638262
Percentage of NaNs in covered_recipient_npi: 72.57%


In [122]:
del dme_year_npi

### LEIE List

In [123]:
# Check the newly created CSV
leie = pd.read_csv("/dsa/groups/casestudycf25/team02/leie_with_valid_npi_clean.csv")

leie.shape

(1739, 18)

In [124]:
leie["npi"] = pd.to_numeric(leie["npi"], errors = "coerce").astype("Int64")

In [126]:
leie["excldate"] = pd.to_datetime(leie["excldate"])      # parse as datetime (if not already)
leie["year_leie"] = leie["excldate"].dt.year

In [127]:
leie.head(1)

,general,specialty,npi,excltype,excldate,num_exclusions_alltime,num_exclusion_types_alltime,list_exclusion_types_alltime,num_addresses_alltime,new_id,fraud_flag,excl_1128a1,excl_1128a2,excl_1128a3,excl_1128b5,excl_1128b6,excl_1128b7,excl_brch cia,year_leie
0,IND- LIC HC SERV PRO,DENTIST,1861529091,1128a3,2024-03-20,1.0,1.0,['1128a3'],1.0,AAMIR-WAHAB-nan-1979-11-16,1,0,0,1,0,0,0,0,2024


In [142]:
leie =  leie[["npi", "year_leie", "excltype", "num_exclusions_alltime","num_exclusion_types_alltime","fraud_flag"]].copy()

In [143]:
dme_payments_leie = dme_payments_merge.merge(
    leie,
    on="npi",
    how="left",
    validate = "many_to_one"
)

In [144]:
dme_payments_leie.shape

(879531, 165)

In [145]:
# condition: dme_year <= leie_year
cond = dme_payments_leie["year"] <= dme_payments_leie["year_leie"]

# find which columns came from leie
leie_cols = [c for c in leie.columns if c != "npi"]

# for rows where the condition fails, blank out the leie columns
dme_payments_leie.loc[~cond, leie_cols] = np.nan

In [146]:
dme_payments_leie["fraud_flag"] = dme_payments_leie["fraud_flag"].fillna(0).astype("Int64")

In [147]:
dme_payments_leie.shape

(879531, 165)

In [148]:
dme_payments_leie.head(1)

,npi,year,target,avg_suplr_sbmtd_chrg_mean,avg_suplr_mdcr_alowd_amt_mean,avg_suplr_mdcr_pymt_amt_mean,avg_suplr_mdcr_stdzd_amt_mean,avg_suplr_sbmtd_chrg_sum,avg_suplr_mdcr_alowd_amt_sum,avg_suplr_mdcr_pymt_amt_sum,...,total_invested_usd,total_value_of_interest,n_owner_records,immediate_family_member,physician_covered_recipient,year_leie,excltype,num_exclusions_alltime,num_exclusion_types_alltime,fraud_flag
0,1003000126,2021,0,129.776563,41.940392,31.813932,34.260562,519.10625,167.761567,127.25573,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [149]:
del dme_payments_merge

### Specialties

In [109]:
# Check the newly created CSV
specialty_npis = pd.read_csv("/dsa/groups/casestudycf25/team02/one_specialty_per_npi.csv")

specialty_npis.shape

(884642, 6)

In [110]:
specialty_npis = specialty_npis[["Rfrg_NPI", "specialty_type", "specialty_lvl1", "specialty"]].copy()
specialty_npis

,Rfrg_NPI,specialty_type,specialty_lvl1,specialty
0,1992997787,Other,Physician Assistants & Advanced Practice Nursi...,Physician Assistant
1,1144628355,Other,Physician Assistants & Advanced Practice Nursi...,Physician Assistant
2,1144656935,Other,Physician Assistants & Advanced Practice Nursi...,Physician Assistant
3,1144726290,Other,Physician Assistants & Advanced Practice Nursi...,Physician Assistant
4,1144777293,Other,Physician Assistants & Advanced Practice Nursi...,Physician Assistant
...,...,...,...,...
884637,1225689060,Doctor of Dentistry,Dental Providers,Dental Therapist
884638,1043558984,Doctor of Dentistry,Dental Providers,Dental Therapist
884639,1265726863,Doctor of Dentistry,Dental Providers,Dental Laboratory Technician
884640,1356443675,Medical Doctor,Allopathic & Osteopathic Physicians,Clinical Pharmacology


In [150]:
dme_payments_leie_specialty = dme_payments_leie.merge(
    specialty_npis,
    left_on="npi",
    right_on="Rfrg_NPI",
    how="inner",
    validate = "many_to_one"
)

In [151]:
########################################################
# CHECK LATER BC ITS MISSING SOME CLAIMS
#########################################################
dme_payments_leie_specialty.shape

(873538, 169)

In [153]:
del dme_payments_leie

In [154]:
dme_payments_leie_specialty.to_csv("/dsa/groups/casestudycf25/team02/unified_dataset.csv", index = False)

In [155]:
dme_payments_leie_specialty["fraud_flag"].value_counts()

0    873334
1       204
Name: fraud_flag, dtype: int64

### Medicare Enrollment

In [99]:
medicare_enrollment = pd.read_csv("/dsa/groups/casestudycf25/team02/medicare_enrollment_clean.csv")

medicare_enrollment.shape

(9671, 26)

In [100]:
medicare_enrollment.head()

,year,bene_state_abrvtn,bene_state_desc,bene_county_desc,bene_fips_cd,tot_benes,orgnl_mdcr_benes,ma_and_oth_benes,aged_tot_benes,dsbld_tot_benes,...,age_70_to_74_benes,age_75_to_79_benes,age_80_to_84_benes,age_85_to_89_benes,age_90_to_94_benes,age_gt_94_benes,dual_tot_benes,b_tot_benes,b_orgnl_mdcr_benes,b_ma_and_oth_benes
0,2021,AL,Alabama,Autauga County,1001,11396.0,5338.0,6059.0,9031.0,2365.0,...,2442.0,1845.0,1111.0,621.0,230.0,55.0,2083.0,10608.0,4549.0,6059.0
1,2021,AL,Alabama,Baldwin County,1003,57509.0,28252.0,29257.0,50529.0,6980.0,...,14433.0,10311.0,6098.0,3152.0,1285.0,393.0,6373.0,53916.0,24660.0,29256.0
2,2021,AL,Alabama,Barbour County,1005,6200.0,2658.0,3542.0,4816.0,1384.0,...,1381.0,990.0,522.0,320.0,129.0,51.0,1897.0,5915.0,2373.0,3542.0
3,2021,AL,Alabama,Bibb County,1007,4701.0,1851.0,2850.0,3489.0,1212.0,...,976.0,693.0,423.0,222.0,83.0,24.0,1180.0,4431.0,1581.0,2850.0
4,2021,AL,Alabama,Blount County,1009,13226.0,5262.0,7964.0,10700.0,2526.0,...,2988.0,2177.0,1258.0,727.0,281.0,67.0,2375.0,12470.0,4506.0,7964.0


In [113]:
medicare_enrollment.dtypes

year                   int64
bene_state_abrvtn     object
bene_state_desc       object
bene_county_desc      object
bene_fips_cd           int64
tot_benes            float64
dtype: object

In [101]:
medicare_enrollment = medicare_enrollment.iloc[:,0:6].copy()

In [102]:
medicare_enrollment

,year,bene_state_abrvtn,bene_state_desc,bene_county_desc,bene_fips_cd,tot_benes
0,2021,AL,Alabama,Autauga County,1001,11396.0
1,2021,AL,Alabama,Baldwin County,1003,57509.0
2,2021,AL,Alabama,Barbour County,1005,6200.0
3,2021,AL,Alabama,Bibb County,1007,4701.0
4,2021,AL,Alabama,Blount County,1009,13226.0
...,...,...,...,...,...,...
9666,2023,PR,Puerto Rico,Yabucoa Municipio,72151,9079.0
9667,2023,PR,Puerto Rico,Yauco Municipio,72153,9015.0
9668,2023,VI,Virgin Islands,St. Croix Island,78010,9986.0
9669,2023,VI,Virgin Islands,St. John Island,78020,935.0


In [103]:
medicare_enrollment.drop_duplicates(subset = ["year", "bene_fips_cd"], inplace = True)

In [104]:
medicare_enrollment.shape

(9671, 6)